### Student Information
- **Name:** Muhammad Ahmad
- **Roll No:** 34801
- **Subject:** Artificial Neural Networks & Deep Learning
- **Class:** BS - AI (Semester-05)

---

# ARTIFICIAL NEURAL NETWORKS & DEEP LEARNING LAB
## Training Workflow, Validation and Model Performance
### Diabetes Outcome Classification using an Artificial Neural Network (MLP)

| Component | Description |
| :--- | :--- |
| **Dataset** | Pima Indians Diabetes (plotly `diabetes.csv`) – 768 patients, 8 input features + `Outcome` |
| **Model** | Multi-Layer Perceptron (ANN) – `MLPClassifier`, hidden layers (16, 8), ReLU, Adam |
| **Target** | `Outcome` (0 = Non-diabetic, 1 = Diabetic) |
| **Preprocessing** | Invalid zeros → missing values, median imputation, `StandardScaler` |
| **Splits** | 80/20 Train-Test, then 75/25 Train-Validation (≈ 60 / 20 / 20 overall) |
| **Validation** | Validation accuracy, 5-Fold Stratified K-Fold, cross-validation score |
| **Performance** | Confusion matrix, precision, recall, F1-score, accuracy, ROC-AUC |
| **Overfitting check** | Train vs validation accuracy, learning curves |
| **Libraries** | pandas, NumPy, Matplotlib, scikit-learn |

**Basis:** This notebook follows the operation list given in the assignment instructions: Data Loading → Preprocessing → Split → Training → Validation → Performance Check → Overfitting Check.

---

### Prerequisites & Asset Setup
- **Required File:** `diabetes.csv`
- **Source:** `https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv`. The loading cell downloads it automatically when internet access is available.
- **Repository Path (offline fallback):** `./assets/diabetes.csv`, i.e. the lab's `assets/` folder in the repository. If the download fails, the loading cell reads this local copy instead.
- **Packages:** `pip install pandas numpy matplotlib scikit-learn` (all pre-installed in Google Colab).

## 1. Learning Objectives & Workflow
- Load the diabetes dataset and check its shape and null values.
- Handle missing values and scale the features with `StandardScaler`.
- Create a Train-Test split (80/20) and a Train-Validation split (75/25).
- Train an Artificial Neural Network with `model.fit(X_train, y_train)`.
- Measure validation accuracy, apply 5-Fold cross-validation and report the cross-validation score.
- Predict `X_test` and evaluate with the confusion matrix, precision, recall, F1-score, accuracy and ROC-AUC curve.
- Check for overfitting by comparing train vs validation accuracy and plotting learning curves.

| Step | Operation (from the assignment) | Notebook section |
| :--- | :--- | :--- |
| 1 | Data Loading: load CSV, check shape and null values | Part 1 |
| 2 | Preprocessing: handle missing values, scaling (`StandardScaler`) | Part 2 |
| 3 | Split: Train-Test (80/20), Train-Validation (75/25) | Part 3 |
| 4 | Training: `model.fit(X_train, y_train)` | Part 4 |
| 5 | Validation: validation accuracy, K-Fold (k=5), cross-validation score | Part 5 |
| 6 | Performance Check: predict `X_test`, confusion matrix, metrics, ROC-AUC | Part 6 |
| 7 | Overfitting Check: train vs validation accuracy, learning curve | Part 7 |

## 2. Background Theory

**Artificial Neural Network (MLP).** A Multi-Layer Perceptron stacks layers of neurons. Each neuron computes a weighted sum of its inputs plus a bias and passes it through a non-linear activation function:

$$a = f\left(\sum_i w_i x_i + b\right)$$

Hidden layers use ReLU, $f(z) = \max(0, z)$. The output neuron uses the sigmoid $\sigma(z) = \frac{1}{1 + e^{-z}}$, which gives the probability that `Outcome = 1`. Training minimises the binary cross-entropy (log-loss) with the Adam optimiser, using back-propagation to compute gradients:

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}\left[y_i \log \hat{p}_i + (1 - y_i)\log(1 - \hat{p}_i)\right]$$

**Why scale the features?** The features live on very different ranges (for example `Insulin` goes up to 846 while `DiabetesPedigreeFunction` stays below 2.5). Gradient-based training behaves badly on such inputs, so `StandardScaler` rescales every feature to zero mean and unit variance:

$$z = \frac{x - \mu}{\sigma}$$

**Train / Validation / Test.** The *training* set is used to learn the weights. The *validation* set is used to monitor the model and compare settings. The *test* set stays untouched until the final evaluation.

**K-Fold cross-validation.** The development data is split into $k = 5$ folds. Each fold is used once for validation while the other four train the model, and the $k$ scores are averaged. *Stratified* K-Fold keeps the Diabetic / Non-diabetic ratio the same in every fold. K-Fold estimates how well the model generalises; it does not make the model itself more accurate.

**Evaluation metrics.**

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN} \qquad \text{Precision} = \frac{TP}{TP + FP}$$

$$\text{Recall} = \frac{TP}{TP + FN} \qquad F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

The **ROC curve** plots the True Positive Rate against the False Positive Rate over all decision thresholds. **AUC** is the area under it: 0.5 means random guessing and 1.0 means perfect separation.

**Overfitting.** A model overfits when training performance is much better than validation performance: it has memorised the training data instead of learning a pattern that generalises. Learning curves (score vs training-set size, or loss vs epoch) make this visible.

## 3. Setup
### 3.1 Import libraries

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    learning_curve
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
    roc_curve,
    log_loss
)

# The ANN is trained for a fixed number of epochs (max_iter=300) on purpose, so that the
# epoch-wise learning curve in Part 7 covers the same training budget. scikit-learn then
# warns that the optimiser has not "converged"; that message is expected and is hidden here.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

SEED = 42
np.random.seed(SEED)
print("Libraries imported successfully.")

## 4. Part 1 – Data Loading
Load the CSV, then check its shape and null values.

### 4.1 Load the dataset

In [ ]:
URL = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
LOCAL_FALLBACK = "assets/diabetes.csv"

try:
    df = pd.read_csv(URL)
    print("Loaded dataset from URL.")
except Exception as e:
    print("URL load failed (" + type(e).__name__ + "). Using local copy:", LOCAL_FALLBACK)
    df = pd.read_csv(LOCAL_FALLBACK)

print("Dataset shape:", df.shape)
display(df.head())

### 4.2 Check shape, data types and null values

In [ ]:
print("Rows    :", df.shape[0])
print("Columns :", df.shape[1])
print("\nColumn names:", df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nNull (NaN) values per column:")
print(df.isnull().sum())
print("\nTotal null values:", df.isnull().sum().sum())

### 4.3 Summary statistics and class distribution

In [ ]:
display(df.describe().T)

print("\nOutcome distribution (0 = Non-diabetic, 1 = Diabetic):")
print(df["Outcome"].value_counts())
print("\nPercentage:")
print(df["Outcome"].value_counts(normalize=True) * 100)

### Observation: Data Loading
- **Shape:** 768 rows × 9 columns, i.e. 8 input features plus the target `Outcome`.
- **Null values:** `isnull().sum()` reports **0** for every column, so there are no explicit NaN values.
- **Hidden problem:** `describe()` shows a **minimum of 0** for `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` and `BMI`. These cannot be zero in a living patient, so the zeros are really *missing values* recorded as 0. Part 2 handles them.
- **Class balance:** 500 Non-diabetic (65.10%) vs 268 Diabetic (34.90%). The data is moderately imbalanced, so accuracy alone is not enough; recall, F1-score and AUC must also be checked.

## 5. Part 2 – Preprocessing
### 5.1 Handle missing values
Zeros in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` and `BMI` are physiologically impossible, so they are converted to `NaN` and later filled with the **median** (robust to skewed columns such as `Insulin`). `Pregnancies = 0` is valid and is left unchanged.

In [ ]:
zero_invalid_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

print("Zero counts BEFORE conversion:")
print((df[zero_invalid_cols] == 0).sum())

df_clean = df.copy()
df_clean[zero_invalid_cols] = df_clean[zero_invalid_cols].replace(0, np.nan)

print("\nMissing (NaN) values AFTER conversion:")
print(df_clean.isnull().sum()[lambda s: s > 0])

print("\nMissing percentage:")
print((df_clean[zero_invalid_cols].isnull().mean() * 100).round(2))

### 5.2 Define imputation and scaling (StandardScaler)

> **Important – avoiding data leakage:** the imputer and the scaler are *defined* here, but they are *fitted on the training data only* after the split in Part 3. The validation and test sets are only *transformed* with the training statistics ($\mu$, $\sigma$, medians). Fitting them on the full dataset would leak information from the validation/test samples into training.

In [ ]:
X = df_clean.drop(columns="Outcome")
y = df_clean["Outcome"]

# Preprocessing: median imputation followed by StandardScaler
preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("X shape:", X.shape)
print("y shape:", y.shape)
print(preprocess)

### Observation: Preprocessing

| Column | Zeros treated as missing | Percentage of rows |
| :--- | :--- | :--- |
| `Glucose` | 5 | 0.65% |
| `BloodPressure` | 35 | 4.56% |
| `SkinThickness` | 227 | 29.56% |
| `Insulin` | 374 | 48.70% |
| `BMI` | 11 | 1.43% |

`Insulin` and `SkinThickness` are missing in a large share of rows, which is why a robust statistic (the median) is used instead of the mean.

## 6. Part 3 – Train-Test and Train-Validation Split
1. **Train-Test split (80/20):** the 20% test set is kept untouched until Part 6.
2. **Train-Validation split (75/25):** the remaining 80% is split again, giving about 60% training, 20% validation and 20% test overall.

Both splits are *stratified*, so the Diabetic / Non-diabetic ratio is the same in every subset.

### 6.1 Create the splits

In [ ]:
# Step 1: Train-Test split (80 / 20)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# Step 2: Train-Validation split (75 / 25 of the 80 %)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=SEED, stratify=y_train_full
)

total = len(X)
print("Total samples         :", total)
print(f"Train + Val (80%)     : {len(X_train_full)}  ({len(X_train_full) / total * 100:.1f}%)")
print(f"  -> Training         : {len(X_train)}  ({len(X_train) / total * 100:.1f}%)")
print(f"  -> Validation       : {len(X_val)}  ({len(X_val) / total * 100:.1f}%)")
print(f"Test (20%)            : {len(X_test)}  ({len(X_test) / total * 100:.1f}%)")

print("\nDiabetic share (Outcome = 1):")
print(f"  Training   : {y_train.mean() * 100:.2f}%")
print(f"  Validation : {y_val.mean() * 100:.2f}%")
print(f"  Test       : {y_test.mean() * 100:.2f}%")

### 6.2 Fit imputer + scaler on the training set, transform all sets

In [ ]:
X_train_s = preprocess.fit_transform(X_train)   # fit ONLY on training data
X_val_s = preprocess.transform(X_val)
X_test_s = preprocess.transform(X_test)

print("Scaled shapes:", X_train_s.shape, X_val_s.shape, X_test_s.shape)
print("\nTraining feature means (should be ~0):")
print(np.round(X_train_s.mean(axis=0), 3))
print("\nTraining feature std (should be ~1):")
print(np.round(X_train_s.std(axis=0), 3))
print("\nAny NaN left in the scaled data?",
      np.isnan(X_train_s).any() or np.isnan(X_val_s).any() or np.isnan(X_test_s).any())

## 7. Part 4 – Training
The ANN is a Multi-Layer Perceptron with the architecture below.

| Layer | Neurons | Activation |
| :--- | :--- | :--- |
| Input | 8 (features) | – |
| Hidden 1 | 16 | ReLU |
| Hidden 2 | 8 | ReLU |
| Output | 1 | Sigmoid (probability of `Outcome = 1`) |

Training runs for a fixed budget of **300 epochs** (`max_iter=300`) with the Adam optimiser.

### 7.1 Build and train the model: `model.fit(X_train, y_train)`

In [ ]:
def build_mlp():
    # Same ANN configuration is reused everywhere in this notebook
    return MLPClassifier(
        hidden_layer_sizes=(16, 8),
        activation="relu",
        solver="adam",
        alpha=1e-3,
        learning_rate_init=1e-3,
        max_iter=300,          # fixed training budget (epochs)
        random_state=SEED
    )

def build_pipeline():
    # Full pipeline (imputer -> scaler -> ANN); used for K-Fold and learning curves
    # so that preprocessing is re-fitted inside every training fold (no leakage).
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("mlp", build_mlp())
    ])

model = build_mlp()
model.fit(X_train_s, y_train)

print("Training finished.")
print("Epochs (iterations) run  :", model.n_iter_)
print("Final training loss      :", round(model.loss_, 4))
print("Layer sizes (in/hidden/out):", [X_train_s.shape[1], *model.hidden_layer_sizes, model.n_outputs_])
print("Output activation        :", model.out_activation_)

### 7.2 Count the trainable parameters

In [ ]:
n_weights = sum(w.size for w in model.coefs_)
n_biases = sum(b.size for b in model.intercepts_)

print("Weights   :", n_weights, "(8x16 + 16x8 + 8x1)")
print("Biases    :", n_biases, "(16 + 8 + 1)")
print("Total trainable parameters:", n_weights + n_biases)

## 8. Part 5 – Validation
### 8.1 Validation accuracy
The model never saw the validation set during training, so its accuracy there is an honest first estimate of generalisation.

In [ ]:
train_pred = model.predict(X_train_s)
val_pred = model.predict(X_val_s)

train_acc = accuracy_score(y_train, train_pred)
val_acc = accuracy_score(y_val, val_pred)

print(f"Training accuracy   : {train_acc:.4f}  ({train_acc * 100:.2f}%)")
print(f"Validation accuracy : {val_acc:.4f}  ({val_acc * 100:.2f}%)")

### 8.2 K-Fold cross-validation (k = 5)
K-Fold is applied to the 80% development data (`X_train_full`). The 20% test set stays untouched. Every fold builds a *fresh* pipeline, so the imputer and scaler are fitted only on that fold's training part.

In [ ]:
k = 5
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=SEED)

fold_scores = []
for fold, (train_index, val_index) in enumerate(skf.split(X_train_full, y_train_full), start=1):
    X_fold_train = X_train_full.iloc[train_index]
    y_fold_train = y_train_full.iloc[train_index]
    X_fold_val = X_train_full.iloc[val_index]
    y_fold_val = y_train_full.iloc[val_index]

    fold_model = build_pipeline()
    fold_model.fit(X_fold_train, y_fold_train)

    score = accuracy_score(y_fold_val, fold_model.predict(X_fold_val))
    fold_scores.append(score)
    print(f"Fold {fold} | Train = {len(train_index)} | Validation = {len(val_index)} | "
          f"Validation Accuracy: {score:.3f}")

print(f"\nMean K-Fold accuracy : {np.mean(fold_scores):.3f}")
print(f"Std                  : {np.std(fold_scores):.3f}")

### 8.3 Cross-validation score with `cross_val_score`
The same 5 folds computed in one line. The result must match the manual loop above.

In [ ]:
cv_scores = cross_val_score(
    build_pipeline(), X_train_full, y_train_full, cv=skf, scoring="accuracy"
)

print("Cross-validation scores:", np.round(cv_scores, 3))
print(f"Mean CV accuracy       : {cv_scores.mean():.3f}")
print(f"Std                    : {cv_scores.std():.3f}")
print(f"95% range (mean +/- 2 std): {cv_scores.mean() - 2 * cv_scores.std():.3f} "
      f"to {cv_scores.mean() + 2 * cv_scores.std():.3f}")
print("\nManual K-Fold matches cross_val_score:", np.allclose(fold_scores, cv_scores))

plt.figure(figsize=(7, 4))
plt.bar(range(1, k + 1), cv_scores, color="steelblue")
plt.axhline(cv_scores.mean(), color="red", linestyle="--", label=f"Mean = {cv_scores.mean():.3f}")
plt.ylim(0.5, 1.0)
plt.xlabel("Fold")
plt.ylabel("Validation accuracy")
plt.title("5-Fold Cross-Validation Accuracy")
plt.xticks(range(1, k + 1))
plt.legend()
plt.show()

### Observation: Validation
- The **validation accuracy** comes from one particular 154-sample split, so it depends on which patients landed in that split.
- **K-Fold** validates on all development samples exactly once. Folds differ from each other (the spread is the *Std* printed above), which shows how much a single split can mislead.
- The **mean cross-validation score** is the more reliable estimate of how the ANN will behave on unseen patients. Do not pick the best-scoring fold as "the" result.
- K-Fold evaluates the model. It does not change the trained weights and does not by itself raise accuracy.

## 9. Part 6 – Performance Check on the Test Set
### 9.1 Predict `X_test`

In [ ]:
y_pred = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)[:, 1]   # probability of Outcome = 1

print("First 20 predictions:", y_pred[:20])
print("First 20 actual     :", y_test.values[:20])
print("\nFirst 5 predicted probabilities:", np.round(y_proba[:5], 3))

### 9.2 Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(cm)
print(f"\nTN (Non-diabetic -> Non-diabetic): {tn}")
print(f"FP (Non-diabetic -> Diabetic)    : {fp}")
print(f"FN (Diabetic -> Non-diabetic)    : {fn}")
print(f"TP (Diabetic -> Diabetic)        : {tp}")

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Non-diabetic", "Diabetic"]
)
disp.plot(cmap="Blues")
plt.title("ANN - Test Confusion Matrix")
plt.show()

### 9.3 Precision, Recall, F1-score and Accuracy

In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy  : {acc:.4f}  = (TP + TN) / total = ({tp} + {tn}) / {len(y_test)}")
print(f"Precision : {prec:.4f}  = TP / (TP + FP)    = {tp} / ({tp} + {fp})")
print(f"Recall    : {rec:.4f}  = TP / (TP + FN)    = {tp} / ({tp} + {fn})")
print(f"F1-score  : {f1:.4f}  = harmonic mean of precision and recall")

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Non-diabetic", "Diabetic"]))

### 9.4 ROC-AUC curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

print(f"ROC-AUC score: {auc:.4f}")

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="darkorange", linewidth=2, label=f"ANN (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], color="gray", linestyle="--", label="Random guess (AUC = 0.5)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve - Diabetes ANN (Test Set)")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

### Observation: Performance Check
- **Confusion matrix:** TN and TP are correct decisions. **FN** (a diabetic patient predicted as non-diabetic) is the costliest error in a screening problem, because a patient who needs follow-up is missed. **FP** only causes an unnecessary extra test.
- **Accuracy vs recall:** because 65% of the patients are Non-diabetic, a reasonable-looking accuracy can hide a weak recall on the Diabetic class. Read the *Diabetic* row of the classification report, not just the overall accuracy.
- **Precision vs recall:** precision answers "of the patients flagged diabetic, how many truly are?", recall answers "of the truly diabetic patients, how many were found?". **F1** balances the two.
- **ROC-AUC:** it measures how well the ANN *ranks* diabetic patients above non-diabetic ones at every threshold, independent of the 0.5 cut-off. The closer the curve hugs the top-left corner, the better; the dashed diagonal is random guessing.

## 10. Part 7 – Overfitting Check
### 10.1 Compare train vs validation accuracy

In [ ]:
train_acc = accuracy_score(y_train, model.predict(X_train_s))
val_acc = accuracy_score(y_val, model.predict(X_val_s))
test_acc = accuracy_score(y_test, model.predict(X_test_s))
gap = train_acc - val_acc

print(f"Training accuracy   : {train_acc:.4f}")
print(f"Validation accuracy : {val_acc:.4f}")
print(f"Test accuracy       : {test_acc:.4f}")
print(f"Train - Validation gap: {gap * 100:+.2f} percentage points")

if gap > 0.10:
    verdict = "OVERFITTING: training accuracy is much higher than validation accuracy."
elif gap > 0.05:
    verdict = "MILD OVERFITTING: a noticeable gap between training and validation accuracy."
elif gap < -0.05:
    verdict = "Validation is clearly better than training - check the split (likely a lucky validation set)."
else:
    verdict = "GOOD FIT: training and validation accuracy are close."
print("\nVerdict:", verdict)

plt.figure(figsize=(6, 4))
bars = plt.bar(["Train", "Validation", "Test"], [train_acc, val_acc, test_acc],
               color=["seagreen", "steelblue", "indianred"])
plt.bar_label(bars, labels=[f"{v:.3f}" for v in [train_acc, val_acc, test_acc]])
plt.ylim(0.5, 1.0)
plt.ylabel("Accuracy")
plt.title("Train vs Validation vs Test Accuracy")
plt.show()

### 10.2 Learning curve: score vs training-set size
The pipeline is trained on growing fractions of the development data. Each point is evaluated with 5-Fold cross-validation.
- **Large gap between the curves that stays open:** high variance (overfitting). More data or a simpler model helps.
- **Both curves low and close together:** high bias (underfitting). A bigger model or better features help.

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    build_pipeline(),
    X_train_full, y_train_full,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=skf,
    scoring="accuracy"
)

train_mean, train_std = train_scores.mean(axis=1), train_scores.std(axis=1)
val_mean, val_std = val_scores.mean(axis=1), val_scores.std(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, "o-", color="seagreen", label="Training accuracy")
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, color="seagreen", alpha=0.15)
plt.plot(train_sizes, val_mean, "o-", color="steelblue", label="Cross-validation accuracy")
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, color="steelblue", alpha=0.15)
plt.xlabel("Number of training samples")
plt.ylabel("Accuracy")
plt.title("Learning Curve - Accuracy vs Training Set Size")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

print("Training samples :", train_sizes)
print("Train accuracy   :", np.round(train_mean, 3))
print("CV accuracy      :", np.round(val_mean, 3))
print(f"\nGap at full size : {(train_mean[-1] - val_mean[-1]) * 100:.2f} percentage points")

### 10.3 Learning curve: loss and accuracy vs epoch
This is the deep-learning style learning curve. The ANN is trained one epoch at a time with `partial_fit`, and the log-loss and accuracy on the training and validation sets are recorded after every epoch. If the **validation loss starts rising while the training loss keeps falling**, the network has begun to memorise the training data.

In [ ]:
EPOCHS = 300
curve_model = build_mlp()
classes = np.unique(y_train)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(EPOCHS):
    curve_model.partial_fit(X_train_s, y_train, classes=classes)

    history["train_loss"].append(log_loss(y_train, curve_model.predict_proba(X_train_s)))
    history["val_loss"].append(log_loss(y_val, curve_model.predict_proba(X_val_s)))
    history["train_acc"].append(curve_model.score(X_train_s, y_train))
    history["val_acc"].append(curve_model.score(X_val_s, y_val))

epochs = np.arange(1, EPOCHS + 1)
best_epoch = int(np.argmin(history["val_loss"])) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history["train_loss"], label="Training loss", color="seagreen")
axes[0].plot(epochs, history["val_loss"], label="Validation loss", color="steelblue")
axes[0].axvline(best_epoch, color="red", linestyle="--", label=f"Lowest val. loss (epoch {best_epoch})")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Log-loss")
axes[0].set_title("Loss vs Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history["train_acc"], label="Training accuracy", color="seagreen")
axes[1].plot(epochs, history["val_acc"], label="Validation accuracy", color="steelblue")
axes[1].axvline(best_epoch, color="red", linestyle="--", label=f"Lowest val. loss (epoch {best_epoch})")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy vs Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Lowest validation loss : {min(history['val_loss']):.4f} at epoch {best_epoch}")
print(f"Validation loss at epoch {EPOCHS}: {history['val_loss'][-1]:.4f}")
print(f"Training loss at epoch {EPOCHS}  : {history['train_loss'][-1]:.4f}")

if history["val_loss"][-1] > min(history["val_loss"]) + 0.01 and best_epoch < EPOCHS:
    print(f"\nValidation loss rises after epoch {best_epoch} while training loss keeps falling -> "
          "the network is overfitting; early stopping around that epoch would help.")
else:
    print("\nValidation loss has not risen noticeably -> no clear overfitting within "
          f"{EPOCHS} epochs.")

### Observation: Overfitting Check
- **Train vs validation accuracy:** the gap printed in 10.1 is the first indicator. Training accuracy above validation accuracy is normal, because the model has seen the training data. A *large* gap is the warning sign.
- **Learning curve (training-set size):** if the cross-validation curve is still rising at the right-hand end, more data would help; a persistent gap between the two curves means the ANN has more capacity than the available samples can pin down.
- **Learning curve (epochs):** training loss normally keeps falling, while validation loss falls and then flattens or rises. The epoch where validation loss is lowest (red dashed line) is the natural stopping point.
- **Remedies if overfitting is present:** early stopping (`early_stopping=True`), a smaller network, a larger L2 penalty `alpha`, or more data.

## 11. Summary of Results

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Training accuracy",
        "Validation accuracy",
        "5-Fold CV accuracy (mean)",
        "5-Fold CV accuracy (std)",
        "Test accuracy",
        "Test precision (Diabetic)",
        "Test recall (Diabetic)",
        "Test F1-score (Diabetic)",
        "Test ROC-AUC",
        "Train - Validation gap (pp)"
    ],
    "Value": [
        round(train_acc, 4),
        round(val_acc, 4),
        round(cv_scores.mean(), 4),
        round(cv_scores.std(), 4),
        round(acc, 4),
        round(prec, 4),
        round(rec, 4),
        round(f1, 4),
        round(auc, 4),
        round(gap * 100, 2)
    ]
})

display(summary)

### Conclusion
- The diabetes data had **no explicit nulls**, but five columns hid missing values as zeros. They were converted to `NaN`, imputed with the training-set median and scaled with `StandardScaler`, with both steps fitted on training data only.
- The data was split 80/20 into development and test sets, and the development part 75/25 into training and validation sets (≈ 60/20/20), all stratified.
- A 16-8-1 ANN was trained with `model.fit`. **Validation accuracy** gave a first estimate, and **5-Fold cross-validation** gave a more stable estimate together with its spread.
- On the untouched test set the ANN was judged with the confusion matrix, precision, recall, F1-score, accuracy and **ROC-AUC**. Because the classes are imbalanced (65% / 35%), recall on the Diabetic class matters more than accuracy alone.
- The **overfitting check** (train vs validation gap, size-based and epoch-based learning curves) shows whether the network generalises or merely memorises, and where early stopping would help.